# Module 1.1: Hardware Evaluation, Safe Battery Removal & 24/7 AC Power Bypass

## Homelab Expansion (Chapter 2) - Step 1: Hardware Refurbishment & Battery Bypass

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand the severe fire and physical damage hazards ("spicy pillow" phenomenon) of leaving a degraded Lithium-Polymer (Li-Po) battery plugged in 24/7.
2. Assemble the complete hardware repair, battery bypass, DC jack stabilization, and thermal maintenance shopping list for a 2019 ASUS ROG Zephyrus (`GA502` / `GU502` / `GX502`).
3. Safely open the Zephyrus chassis, disconnect the motherboard battery connector without shorting nearby SMD capacitors, and extract the degraded battery pack.
4. Diagnose and fix **DC barrel jack micro-disconnects** (where bumping the table causes an instant power-off) via spring-leaf retention adjustment, chassis strain-relief anchoring, or an optional inline `XT60` lock.
5. Insulate the exposed battery header, clear the ASUS Embedded Controller (EC) "dead boot" latch, and mitigate missing battery transient power bridging using an external UPS.

## 1.1.1 Conceptual Background: Why Remove & Bypass a Degraded Laptop Battery?

When repurposing a 2019 gaming laptop like the **ASUS ROG Zephyrus** (`GA502`, `GU502`, or `GX502`, equipped with an `C41N1837` / `76Wh` 4-cell Li-Po pouch battery) into an always-on 24/7 server, leaving a worn-out battery inside the chassis is the single biggest reliability and safety hazard in a homelab.

### 1. The Lithium-Polymer "Spicy Pillow" Hazard
Unlike cylindrical `18650` cells, thin gaming laptops use soft aluminum-laminated pouch cells. When a degraded pouch battery is held at 100% trickle charge inside a warm chassis running 24/7 workloads, electrolyte decomposition generates flammable gases ($\text{CO}_2$, $\text{CO}$, and hydrocarbons). This causes the pouch to swell:
- **Mechanical Damage:** On the 2019 ROG Zephyrus, the battery sits directly beneath the trackpad and palmrest. A swelling battery bends the magnesium-alloy deck, cracks the trackpad glass, and stresses the motherboard PCB traces.
- **Thermal Runaway:** A punctured or severely swollen Li-Po pouch can enter thermal runaway. Removing the battery entirely eliminates this hazard permanently.

### 2. How the Laptop "Battery Bypass" Actually Works
Modern laptops use a buck-boost DC-DC charging IC (Narrow VDC / Hybrid Power Boost architecture) controlled by the motherboard's **Embedded Controller (EC)**:

```mermaid
flowchart LR
    UPS["External Pure Sine-Wave UPS<br>(Replaces Internal Battery)"] ==>|"120V AC"| Brick["OEM ASUS AC Power Brick<br>(180W / 230W DC Barrel)"]
    Brick ==>|"19.5V - 20V DC (Strain-Relieved)"| ChargerIC["Motherboard Charger IC<br>& Power Rail MOSFETs"]
    
    subgraph Laptop ["2019 ROG Zephyrus Motherboard"]
        ChargerIC ==>|"Direct Primary System Rail (VSYS)"| VRM["CPU & NVIDIA GPU VRMs"]
        BattHeader["Internal Battery Header<br>(Unplugged & Kapton-Insulated)"] -. "Open Circuit (Bypassed)" .- ChargerIC
        EC["Embedded Controller (EC)<br>(Monitors Adapter Wattage)"] --> VRM
    end
```

- **No Solder Jumpers Required:** You do **not** solder wires across the battery terminals! Soldering across a lithium battery connector will dead-short the charging MOSFETs and destroy the motherboard. The Zephyrus motherboard natively routes 19.5V/20V DC from the barrel jack straight to the `VSYS` system power rail when the battery connector is left open (unplugged).
- **The Three Engineering Caveats of Running Battery-Less:**
  1. **Zero Hold-Up Capacitance Against DC Barrel Vibration ("Knee-Ding" Shutoffs):** A healthy battery normally acts as an inline UPS when a loose barrel plug loses contact for 5 milliseconds after you bump the table. Once the battery is dead or removed, motherboard bulk capacitors only hold up the `VSYS` rail for **<2 milliseconds**—so *any* micro-wiggle in the `6.0mm x 3.7mm` barrel jack causes an instant hard power-off. We solve this permanently in **Task 1.1.3** below!
  2. **The ASUS EC "Dead Boot" Quirk:** The first time AC power is plugged in after removing the battery, the Embedded Controller may refuse to power on because its CMOS/EC state registers a missing battery fault. Performing a **45-second EC Hard Reset** clears this latch.
  3. **Transient Power Bridging (Hybrid Power Boost):** Under extreme gaming spikes, ASUS laptops briefly pull 10W–20W from the battery to assist the AC brick. Without a battery, sudden 100% CPU + 100% GPU spikes can trip the AC adapter's Over-Current Protection (OCP). We solve this cleanly in software (capping the NVIDIA GPU power limit by 15%–20% via `nvidia-smi -pl` in Module 4.1) and externally via a **UPS**.

## 🛒 Hardware Repair, DC Jack Lock & Battery Bypass Shopping List

Before opening the laptop, make sure you have the following tools and consumables on your workbench:

| Category | Item & Specification | Why You Need It |
| :--- | :--- | :--- |
| **Driver Kit** | **Precision Phillips `PH0` & `PH1` Screwdriver + Plastic Spudger** | Removes the 14–15 bottom cover screws and 4–6 internal battery frame screws without stripping heads or scratching the chassis. |
| **Insulation** | **10mm Kapton (Polyimide) Electrical Tape** | Covers the exposed motherboard battery connector pins and secures any loose sensor cables after battery extraction. |
| **DC Jack Lock** | **Adhesive Nylon P-Clips / Zip-Tie Mounts + Heavy-Duty Adhesive Velcro Strips** *(Optional: Solder-Reflow Iron or Amass `XT60H` Pair)* | Eliminates downward cord torque from the heavy ASUS power brick and mechanically locks the barrel cord in a zero-tension loop against the chassis so bumping the desk can never unseat the plug. |
| **ESD Safety** | **Nylon / ESD-Safe Angled Tweezers** | Safely slides back the metal locking bail on the ASUS battery connector and adjusts DC barrel spring leaves (when unplugged). |
| **Thermal Refresh** | **Arctic MX-6 / Noctua NT-H2 (or PTM7950) + `K5 PRO` Viscous Thermal Putty + 99% Isopropyl Alcohol** | While the bottom cover is off (Module 1.2), you will clean 7 years of dust/oxidation and replace dried 2019 factory paste and VRM thermal putty. |
| **External Bypass** | **600VA–1000VA Pure Sine-Wave UPS** *(e.g., CyberPower CP1000PFCLCD or APC Back-UPS)* | Acts as your external, fire-safe battery outside the laptop chassis—preventing sudden power-loss crashes and dirty AC brownouts. |

## 1.1.2 Step-by-Step Laboratory Guide

### Task 1.1.1: Pre-Teardown Safety & Screw Mapping
1. Shut down the ROG Zephyrus completely.
2. Unplug the OEM AC power barrel adapter and remove all USB peripherals.
3. Press and hold the **Power Button for 15 seconds** to discharge residual capacitance in the motherboard rails.
4. Flip the laptop upside down onto an anti-static mat or clean microfiber towel.
5. **Crucial Zephyrus Screw Map:** On 2019 Zephyrus models (`GA502` / `GU502` / `GX502`), the bottom panel uses **different screw lengths** (longer screws along the rear hinge/exhaust vents, shorter screws along the front lip, and often one captive pop-up screw on the bottom-right corner). Draw a quick grid on a sheet of paper and place each screw on its matching spot as you remove it.
6. Use your plastic spudger starting at the captive corner gap to gently pop the perimeter clips and lift away the bottom panel.

### 🖼️ Task 1.1.1 Verification: Initial Internal Hardware Inspection

**Verification Objective:** Capture a top-down photo of the opened ROG Zephyrus interior showing the original battery pack, dual cooling fans, and copper heatpipes before modification.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.1.2: Safe Battery Connector Release & Pack Extraction

> ⚠️ **CRITICAL MOTHERBOARD WARNING:** The 2019 ASUS ROG Zephyrus uses a vertical/slide-lock 8-pin battery connector surrounded by tiny surface-mount capacitors. **Never use a metal flathead screwdriver to pry the battery plug!** A 1mm slip with metal while the battery is still plugged in will short the 15.4V battery rail directly to ground or the EC data lines, permanently killing the motherboard.

1. **Slide Back the Metal Locking Clip:** Inspect where the multi-wire battery cable meets the motherboard. Using a fingernail or plastic spudger, gently slide the small silver metal retaining tab **horizontally away from the battery plug** (toward the motherboard/heatsink) by ~2mm until it unlatches.
2. **Lift Vertically:** Grab the black plastic connector housing (or pull evenly on both sides with plastic fingers/spudgers) and lift **straight up** vertically off the motherboard pins.
3. **Unscrew the Battery Frame:** Remove the Phillips screws securing the black `C41N1837` Li-Po battery frame to the palmrest chassis.
4. **Lift & Inspect:** Carefully lift the battery out of the laptop. (If the cells are swollen, do not bend or compress the pouch). Place it in a non-conductive container to take to a certified e-waste/battery drop-off center (e.g., Best Buy, Home Depot Call2Recycle, or municipal household hazardous waste).
5. **Insulate the Header:** Apply a small strip of **Kapton tape** over the exposed motherboard battery connector housing so no stray metallic debris can ever bridge the pins.

### 🖼️ Task 1.1.2 Verification: Empty Battery Bay & Insulated Header

**Verification Objective:** Capture a photo of the laptop interior with the Li-Po battery completely removed and the motherboard battery connector cleanly unlatched/insulated.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 1.1.3: Fixing the "Table Bump / Knee-Ding" Power Cutoff (DC Barrel Jack Hardening)

If bumping the table with your knee causes the laptop to instantly lose power, **the culprit is two-fold**:
1. Your dead internal battery was providing **0.0Wh of hold-up capacity**, so even a **1-millisecond loss of contact** in the DC barrel jack drops the `19.5V VSYS` rail to `0V`.
2. After 7 years of plugging/unplugging (and the heavy ASUS power brick pulling downward on the cord), the outer ground spring leaves inside the `6.0mm x 3.7mm` DC barrel jack relax—or one of the through-hole ground lugs on the PCB develops a hairline ring crack.

While the laptop is open and **completely unplugged from AC power**, perform this 4-step inspection and mechanical stabilization to make the DC connection rock-solid:

```mermaid
flowchart TD
    Inspect["1. Inspect DC-In Jack Inside Open Chassis<br>(Left-side edge of motherboard)"] --> CheckWiggle{"Does the black DC jack housing<br>rock against the green PCB?"}
    CheckWiggle -->|"Yes (Cracked Solder Joint)"| Reflow["Reflow the 5 through-hole solder pads<br>with fresh flux & leaded solder<br>(or solder an inline 14AWG XT60 pigtail)"]
    CheckWiggle -->|"No (Housing is solid on PCB)"| Spring["2. Retension Internal Ground Spring Leaves<br>& Clean with 99% IPA"]
    Reflow --> Spring
    Spring --> Anchor["3. Anchor Cable to Chassis & Velcro Brick to Desk<br>(Zero Tension on Barrel Plug)"]
```

#### Step 1: Check for a Cracked PCB Solder Joint vs. Loose Barrel Spring
- Look at the black rectangular **DC-In Barrel Jack** soldered to the left edge of the ROG Zephyrus motherboard.
- Insert the unplugged barrel tip into the jack and wiggle it gently while watching the **back and underside of the DC jack housing on the motherboard**:
  - **Case A (Jack housing stays motionless on the PCB, but the metal plug wiggles inside the barrel):** The solder joints are healthy! The internal ground spring leaves inside the barrel socket have simply flattened out over 7 years. Proceed to **Step 2**.
  - **Case B (The entire black plastic jack housing tilts or lifts slightly off the PCB):** One of the through-hole mechanical ground lugs or the rear positive pin has a cracked solder fillet from years of cord tugging. If you have a soldering iron (`350°C–380°C`), add a dab of rosin flux and reflow fresh solder onto the large through-hole pins on the motherboard until they are shiny and rigid.

#### Step 2: De-Oxidate & Re-Tension the Barrel Ground Spring Leaf (Unplugged!)
1. Confirm the AC power brick is **unplugged from the wall** and the internal battery is **removed**.
2. Dip a cotton swab in **99% Isopropyl Alcohol** and scrub both the inside of the laptop's DC jack (being careful not to snag cotton on the thin center needle pin) and the outer/inner walls of the charger plug to remove carbon micro-arcing residue.
3. Look inside the laptop's DC barrel socket with a flashlight: along the outer perimeter wall (`6.0mm` diameter), you will see a curved **metallic ground tension spring leaf** (or two leaves) designed to press inward against the outer sleeve of the barrel plug.
4. Using a fine pick or small precision flathead jeweler's screwdriver, gently pry the curved ground spring leaf **inward toward the center pin by ~0.5mm** so it regains its factory spring tension.
5. Plug the barrel tip back in—you should immediately feel a much firmer, tactile "snap" and resistance against wobbling!

#### Step 3: Mechanical Strain-Relief Loop ("Knee-Proofing" the Cord)
Even a tight barrel jack will loosen again if the heavy `180W/230W` ASUS power brick hangs off the desk or transfers table vibrations directly to the plug tip:
1. **Anchor the Power Brick with Velcro:** Apply two strips of heavy-duty adhesive Velcro to the smooth side of the ASUS power brick and stick it firmly to the underside or surface of your server shelf/table so the brick's weight never hangs on the DC cord.
2. **Create a Chassis Strain-Relief Loop:** Stick an **adhesive nylon cable clip (or adhesive zip-tie mount)** onto the table or the rear-left corner of the laptop chassis (`2 inches` behind the DC barrel port).
3. Push the right-angle (or straight) barrel plug firmly into the jack, route the cable backward in a gentle U-loop, and cinch it into the adhesive clip.
   - **Result:** If your knee bumps the table or catches the power cord, 100% of the mechanical force stops at the adhesive cable anchor—zero lateral torque ever reaches the barrel jack!

> 🔧 **Advanced Permanent Option (Only if your DC jack is physically broken beyond spring adjustment):** If the center needle inside your laptop's barrel jack is physically snapped off, don't try to wire a generic non-ASUS power supply—instead, either solder a `$6` replacement ASUS `6.0mm x 3.7mm` 7-pin DC jack onto the board, **or** solder two short `14 AWG` silicone leads (`+19.5V` to the positive PCB pad, `-` to the ground lug) terminating in a locking **Amass `XT60H` connector** outside the vent slot, and put the mating `XT60H` plug onto the output cord of your **original OEM ASUS power brick**. That keeps the exact OEM voltage regulation and protection while giving you an industrial locking plug that takes 30 lbs of pull force to disconnect!

### Task 1.1.4: Embedded Controller (EC) Reset & Battery-Less AC Boot + "Bump Test"
Before proceeding to thermal maintenance in Module 1.2, verify that the laptop boots cleanly on pure AC wall power and passes a desk-vibration test:
1. Temporarily set the bottom cover back in place (no need to tighten all screws yet, as we will service the heatsinks in Module 1.2) and flip the laptop right-side up.
2. **Perform the ASUS EC Reset (with AC Unplugged):** Hold down the **Power Button continuously for 45 seconds**. This resets the Embedded Controller's battery-detection registers.
3. **Connect the OEM Barrel Charger (via your UPS & Strain-Relief Clip):** Plug the original high-wattage ASUS barrel power brick (180W/230W—do *not* use a 65W USB-C PD charger as your primary power source without a battery!) into the laptop.
4. Press the **Power Button** and immediately tap `F2` (or `DEL`) repeatedly to enter the **ASUS ROG UEFI BIOS Utility**.
5. **Perform the "Knee-Ding" Vibration Test safely inside BIOS:** While sitting on the BIOS screen (where no disk writes can be corrupted), gently tap the table leg and lightly wiggle the strain-relieved power cord behind the anchor clip to confirm the barrel jack maintains rock-solid electrical contact without dropping power.

### 🖼️ Task 1.1.4 Verification: Battery-Less AC Boot & Strain-Relieved Barrel Jack

**Verification Objective:** Capture a photo of the ROG Zephyrus booted into the ASUS UEFI BIOS Utility running exclusively on the strain-relieved AC power adapter with the internal battery removed.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification photo below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] Bottom panel screws mapped by position and length.
- [ ] Degraded Li-Po battery safely unlatched, unscrewed, removed, and staged for certified e-waste recycling.
- [ ] Motherboard battery header inspected and covered with Kapton tape.
- [ ] DC barrel jack solder joints verified, internal ground spring leaf cleaned & retensioned, and cord anchored with strain relief.
- [ ] EC reset completed and battery-less AC boot + table vibration test in ASUS UEFI BIOS verified.
- [ ] Verification photos captured and attached above.

Proceed to [Module 1.2: Deep Thermal Maintenance & BIOS Server Tuning](./01_2_thermal_maintenance_and_bios_prep.ipynb).